# 05 - Trade Attribution

Per-asset attribution of confirmed swaps using
`guardrail_lab.attribution`. Each `tx_confirmed` is correlated with
its preceding `order_proposed` (destination symbol) and risk
decision (final executed amount).

**Offline-safe:** an empty log prints a hint.


In [ ]:
# --- Guardrail Alpha notebook bootstrap (offline-safe) ---
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Return the first ancestor that contains python-lab/guardrail_lab."""
    for candidate in [start, *start.parents]:
        if (candidate / "python-lab" / "guardrail_lab").is_dir():
            return candidate
    return start


# In a notebook __file__ is undefined, so anchor on the current working dir.
REPO_ROOT = _find_repo_root(Path.cwd())
LAB_PATH = REPO_ROOT / "python-lab"
if str(LAB_PATH) not in sys.path:
    sys.path.insert(0, str(LAB_PATH))


def _first_existing(*candidates: Path):
    """Return the first path that exists, else None."""
    for path in candidates:
        if path.exists():
            return path
    return None


DATA_DIR = REPO_ROOT / "data"
CONFIG_DIR = REPO_ROOT / "configs"

# Prefer a real run; fall back to the seeded demo artifacts if present.
DB_PATH = _first_existing(
    DATA_DIR / "guardrail_alpha.db",
    DATA_DIR / "demo_guardrail_alpha.db",
)
RUN_REPORT_PATH = _first_existing(
    DATA_DIR / "run_report.json",
    DATA_DIR / "demo_run_report.json",
)
EXPERIMENTS_DIR = DATA_DIR / "experiments"
BACKTESTS_DIR = DATA_DIR / "backtests"
ELIGIBLE_ASSETS_PATH = CONFIG_DIR / "eligible_assets.bsc.json"
ASSET_CATEGORIES_PATH = CONFIG_DIR / "asset_categories.json"

NO_DATA_HINT = (
    "No data found under data/. Run the agent (paper/live) or seed a demo run "
    "first, e.g.  python3 -m guardrail_lab.seed  (from python-lab/), then "
    "re-run this notebook. The notebook is offline-safe and will not raise."
)

print("repo root :", REPO_ROOT)
print("event log :", DB_PATH if DB_PATH else "(none yet)")
print("run report:", RUN_REPORT_PATH if RUN_REPORT_PATH else "(none yet)")


In [ ]:
def render_table(rows, columns, title=None):
    """Print an aligned text table. rows: list[dict]; columns: list[(key,label)]."""
    if title:
        print(title)
        print("=" * len(title))
    if not rows:
        print("(no rows)")
        return
    labels = [label for _, label in columns]
    keys = [key for key, _ in columns]
    cells = [[("" if r.get(k) is None else str(r.get(k))) for k in keys] for r in rows]
    widths = [
        max(len(labels[i]), *(len(row[i]) for row in cells)) for i in range(len(keys))
    ]
    header = "  ".join(labels[i].ljust(widths[i]) for i in range(len(keys)))
    print(header)
    print("  ".join("-" * widths[i] for i in range(len(keys))))
    for row in cells:
        print("  ".join(row[i].ljust(widths[i]) for i in range(len(keys))))


## Per-asset attribution


In [ ]:
from guardrail_lab.db import load_events
from guardrail_lab.attribution import trade_attribution, regime_timeline

events = load_events(str(DB_PATH)) if DB_PATH else []
attribution = trade_attribution(events)

if not attribution:
    print(NO_DATA_HINT)
else:
    total_usd = sum(a["total_amount_usd"] for a in attribution)
    total_trades = sum(a["count"] for a in attribution)
    print(f"Confirmed swaps: {total_trades}  |  Total notional: {total_usd:,.2f} USD\n")
    render_table(
        [
            {
                "symbol": a["symbol"],
                "trades": a["count"],
                "notional_usd": f"{a['total_amount_usd']:,.2f}",
                "share": f"{(a['total_amount_usd'] / total_usd * 100.0):.1f}%" if total_usd else "n/a",
            }
            for a in attribution
        ],
        [
            ("symbol", "SYMBOL"),
            ("trades", "TRADES"),
            ("notional_usd", "NOTIONAL_USD"),
            ("share", "SHARE"),
        ],
        title="Per-asset trade attribution",
    )


## Notional sparkline by asset


In [ ]:
def bar(value, max_value, width=30):
    if not max_value:
        return ""
    filled = int(round(value / max_value * width))
    return "█" * filled + "·" * (width - filled)

if attribution:
    max_usd = max(a["total_amount_usd"] for a in attribution)
    for a in attribution:
        print(f"{a['symbol']:>6}  {bar(a['total_amount_usd'], max_usd)}  "
              f"{a['total_amount_usd']:,.2f}")
else:
    print(NO_DATA_HINT)


## Regime timeline (context)


In [ ]:
if events:
    timeline = regime_timeline(events)
    if timeline:
        render_table(
            timeline,
            [("timestamp", "TIMESTAMP"), ("regime", "REGIME")],
            title="Regime classifications over time",
        )
    else:
        print("No regime_classified events in the log yet.")
else:
    print(NO_DATA_HINT)


## Notes

* Notional uses the risk-adjusted amount when a `risk_approved` /
  `risk_clipped` decision is present, otherwise the proposed order
  amount.
* Unknown destinations are bucketed under `UNKNOWN` rather than
  dropped.
